# encoding3d in Python

This notebook reads, inspects and writes encoding3d files with `openalea.encoding3d`, the format's Python reference codec.
It uses the repository's conformance files, small reference scenes in `conformance/`, as examples.

- a **frame** is one time step of a scene: one `.glb` file;
- it holds **data structures** (a plant population, a soil grid...), each made of **entity sets** (one per scale: Plant,
  Axis, SubOrgan...), which are tables of numpy columns;
- every variable column comes with a **declaration**: unit, standard name, kind, role.

In [ ]:
import os

import numpy as np
import pandas as pd

from openalea import encoding3d

CONFORMANCE = os.path.join("..", "..", "conformance")   # the repository's reference scenes

## Reading a frame

`encoding3d.read` gives a `Frame`: its time, and its data structures.

In [ ]:
frame = encoding3d.read(os.path.join(CONFORMANCE, "soil_and_roots.glb"))
print(frame.time, frame.time_unit, frame.step)
for data_structure in frame.data_structures:
    print(data_structure.id, data_structure.kind, [entity_set.name for entity_set in data_structure.entity_sets])

Each entity set is a table: one row per entity, one numpy column per variable. Pandas shows it directly (vector
columns, one per row, aside).

In [ ]:
roots = frame.data_structure("root_system")
segments = roots.entity_set("SubOrgan")
pd.DataFrame({name: values for name, values in segments.columns.items() if np.ndim(values) == 1})

The columns `id`, `label`, `complex`, `parent` and `edge_type` are the multiscale graph: `complex` is the entity
containing a segment one scale up (its axis), `parent` the segment it follows, and `edge_type` how (in line, or as a
branch).

## What the variables mean

Each variable is declared by its writer: its unit (a UCUM code), its standard name in the shared vocabulary, how it
combines (its kind) and its role.

In [ ]:
pd.DataFrame({name: vars(variable) for name, variable in segments.variables.items()}).T[
    ["unit", "standard_name", "kind", "role", "description"]]

In [ ]:
from openalea.encoding3d import vocabulary

vocabulary.standard_names()["water_potential"]["definition"], vocabulary.standard_names()["water_potential"]["canonical_unit"]

## The soil, and which cells each root segment lies in

A grid is a data structure too, and a **mapping** says which soil cell each segment is in, with a weight.

In [ ]:
soil = frame.data_structure("soil")
print(soil.grid)
mapping = frame.mappings[0]
pd.DataFrame(mapping.columns)

## Geometry

A mesh's vertices carry the row of their entity at every scale they represent: here each triangle knows its segment,
its axis and its plant.

In [ ]:
mesh = roots.meshes[0]
print(len(mesh.positions) // 3, "triangles; rows by scale:", list(mesh.feature_rows))
first_triangle_segment = segments.columns["id"][mesh.feature_rows["SubOrgan"][0]]
first_triangle_segment

## A sequence of frames

A simulation writes a frame every so many steps, with an index, `scene.fspm.json`. Entities keep their ids from
frame to frame.

In [ ]:
index = os.path.join(CONFORMANCE, "growing_sequence", "scene.fspm.json")
rows = []
for growing in encoding3d.read_sequence(index):
    elements = growing.data_structures[0].entity_set("SubOrgan")
    rows.append({"time (d)": growing.time, "segments": elements.count, "root length (m)": elements.columns["length"].sum()})
pd.DataFrame(rows)

## Writing and checking a frame

Writing builds the same objects. The validator checks a file against the specification: ids, links, units against
the vocabulary, tables.

In [ ]:
from openalea.encoding3d.validate import validate_bytes

plant = encoding3d.EntitySet("Plant", rank=1, columns=dict(id=np.array([1]), label=np.array(["Wheat"], dtype=object)))
segments = encoding3d.EntitySet("SubOrgan", rank=6, columns=dict(
    id=np.array([10, 11, 12]), label=np.array(["RootSegment"] * 3, dtype=object), complex=np.array([1, 1, 1]),
    parent=np.array([-1, 10, 11]), edge_type=np.array(["none", "successor", "successor"], dtype=object),
    length=np.array([0.01, 0.01, 0.004])),
    variables={"length": encoding3d.Variable(unit="m", standard_name="length", kind="extensive")})
written = encoding3d.Frame([encoding3d.DataStructure("roots", entity_sets=[plant, segments])], time=0., time_unit="d")
data = encoding3d.to_bytes(written)
print(len(data), "bytes;", validate_bytes(data))

In [ ]:
again = encoding3d.from_bytes(data)
again.data_structure("roots").entity_set("SubOrgan").columns["length"]

A wrong unit is caught: a length in seconds does not convert to the standard name's metres.

In [ ]:
segments.variables["length"] = encoding3d.Variable(unit="s", standard_name="length", kind="extensive")
print(validate_bytes(encoding3d.to_bytes(written)))

## For other viewers: display copies

Viewers that do not read the tables (Blender, a web viewer, ParaView without encoding3d's plugin) colour shapes by
per-vertex attributes. A display copy adds one, from the tables.

In [ ]:
from openalea.encoding3d.display import add_display

shown = add_display(encoding3d.read(os.path.join(CONFORMANCE, "root_system.glb")), ["water_potential"])
mesh = shown.data_structures[0].meshes[0]
mesh.display["water_potential"][:6], mesh.display_sources